In [ ]:
!pip install -q transformers datasets

In [ ]:
import torch
import numpy as np
from transformers import CLIPVisionModel, CLIPImageProcessor
from datasets import load_dataset
from torch.utils.data import DataLoader

In [ ]:
device = 'cuda'

In [ ]:
n_train = 10000
n_eval = 500
out = "/content/drive/MyDrive/cadquery"

In [ ]:
import numpy as np, torch
from transformers import AutoTokenizer, CLIPVisionModel, CLIPImageProcessor
from datasets import load_dataset
from torch.utils.data import DataLoader

train = load_dataset("CADCODER/GenCAD-Code", split="train")
test  = load_dataset("CADCODER/GenCAD-Code", split="test")

tok = AutoTokenizer.from_pretrained("gpt2")

def data_filter(ds, n, col="cadquery"):
    texts = ds[col]
    ok = [i for i, c in enumerate(texts) if isinstance(c, str) and c.strip()]
    ds = ds.select(ok)
    texts = ds[col]
    lens = []
    for i in range(0, len(texts), 1000):
        lens.extend(len(x) for x in tok(texts[i:i+1000], truncation=False)["input_ids"])
    keep = np.where(np.array(lens) <= 950)[0]
    return ds.select(keep).shuffle(seed=0).select(range(min(n, len(keep))))

sub_train, sub_eval = data_filter(train, 10000), data_filter(test, 500)

proc = CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32",
        torch_dtype=torch.float16).cuda().eval()

def encode(ds, path):
    def collate(b):
        return (proc(images=[x["image"].convert("RGB") for x in b], return_tensors="pt")["pixel_values"], [x["cadquery"] for x in b])
    feats, codes = [], []
    for px, cd in DataLoader(ds, batch_size=64, collate_fn=collate, num_workers=2):
        with torch.no_grad():
            feats.append(clip(pixel_values=px.cuda().half()).last_hidden_state.cpu())
        codes.extend(cd)
    f = torch.cat(feats); print(path, f.shape)
    torch.save({"feats": f, "codes": codes}, path)


os.makedirs("/content/cadquery", exist_ok=True)
encode(sub_train, "/content/cadquery/train.pt")
encode(sub_eval,  "/content/cadquery/eval.pt")
encode(sub_train, "/content/drive/MyDrive/cadquery/train.pt")
encode(sub_eval,  "/content/drive/MyDrive/cadquery/eval.pt")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1245 > 1024). Running this sequence through the model will result in indexing errors


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_n

/content/cadquery/train.pt torch.Size([10000, 50, 768])
/content/cadquery/eval.pt torch.Size([500, 50, 768])
/content/drive/MyDrive/cadquery/train.pt torch.Size([10000, 50, 768])
/content/drive/MyDrive/cadquery/eval.pt torch.Size([500, 50, 768])


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs("/content/drive/MyDrive/cadquery", exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
d = torch.load("/content/drive/MyDrive/cadquery/train.pt")
print(d["feats"].shape, len(d["codes"]))

torch.Size([10000, 50, 768]) 10000
